In [6]:
import os
import pandas as pd
import numpy as np
from datetime import datetime

# =============================================================================
# 1. 設定
# =============================================================================
BASE_PATH_RESULTS = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/Corr_1000/results"
OUTPUT_ROOT = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/Corr_1000"
NEW_DATE = datetime.now().strftime("%Y%m%d")

# 目的変数の表示順序
target_order = ["PCEmax", "Jsc", "Voc", "FF"]
# データセット名の読み替えルール
dataset_map = {
    "m_all_FP_rebuilt.csv": "500S+FP",
    "n_all_FP_rebuilt.csv": "500S+FP",
    "m_all_OH_rebuilt.csv": "500S+OH",
    "n_all_OH_rebuilt.csv": "500S+OH",
    "m_base_FP_rebuilt.csv": "7変数+FP",
    "n_base_FP_rebuilt.csv": "7変数+FP",
    "m_base_OH_rebuilt.csv": "7変数+OH",
    "n_base_OH_rebuilt.csv": "7変数+OH",
    "m_all.csv": "500S",
    "n_all.csv": "500S",
    "m_base.csv": "7変数",
    "n_base.csv": "7変数"
}

# -----------------------------------------------------------------------------
# 2. 全データの収集とランキング
# -----------------------------------------------------------------------------
summary_list = []
for dname in os.listdir(BASE_PATH_RESULTS):
    spath = os.path.join(BASE_PATH_RESULTS, dname, "all_summary_CV10.csv")
    if os.path.exists(spath):
        df = pd.read_csv(spath)
        # フォルダ名からモデル名を取得（Corr_1000_XXX の XXX 部分）
        model_name = dname.replace("Corr_1000_", "")
        df['Model_Name'] = model_name
        summary_list.append(df)

all_summaries = pd.concat(summary_list, ignore_index=True)
all_summaries['補完有無'] = all_summaries['File'].apply(lambda x: "補完" if x.startswith("m_") else "未補完")
all_summaries['Target'] = all_summaries['Target'].str.strip()

# 各条件で R2 max / RMSE min を検索
best_models = all_summaries.sort_values(
    by=["Target", "補完有無", "CV10_R2", "CV10_RMSE"], 
    ascending=[True, False, False, True] # 補完有無は「補完」から先にしたいのでFalse
).drop_duplicates(subset=["Target", "補完有無"])

# -----------------------------------------------------------------------------
# 3. 指定フォーマットへの整形
# -----------------------------------------------------------------------------
final_table_rows = []

for tgt in target_order:
    for imp in ["補完", "未補完"]:
        row = best_models[(best_models["Target"] == tgt) & (best_models["補完有無"] == imp)]
        
        if not row.empty:
            r = row.iloc[0]
            # 表示名の調整
            tgt_display = "PCE" if tgt == "PCEmax" else tgt
            dataset_display = dataset_map.get(r['File'], r['File'])
            
            final_table_rows.append({
                "目的変数": tgt_display,
                "補完有無": imp,
                "アルゴリズム": r['Model_Name'],
                "データセット": dataset_display,
                "R2（RMSE）": f"{r['CV10_R2']:.3f} （{r['CV10_RMSE']:.3f}）"
            })

# データフレーム化
df_optimal = pd.DataFrame(final_table_rows)

# 目的変数の重複表示を消す処理（見た目の調整：任意）
# df_optimal.loc[df_optimal['目的変数'].duplicated(), '目的変数'] = ""

# CSV保存
output_path = os.path.join(OUTPUT_ROOT, f"Optimal_Model_Selection_Table_{NEW_DATE}.csv")
df_optimal.to_csv(output_path, index=False, encoding='utf-8-sig')

# -----------------------------------------------------------------------------
# 4. コンソールへの表示
# -----------------------------------------------------------------------------
print("\n" + "="*90)
print(f"{'目的変数':<10} | {'補完有無':<6} | {'アルゴリズム':<15} | {'データセット':<12} | {'R2（RMSE）'}")
print("-" * 90)
for _, r in df_optimal.iterrows():
    print(f"{r['目的変数']:<10} | {r['補完有無']:<8} | {r['アルゴリズム']:<15} | {r['データセット']:<12} | {r['R2（RMSE）']}")
print("="*90)

print(f"\n[SUCCESS] 最適モデル表を作成しました:\n{output_path}")


目的変数       | 補完有無   | アルゴリズム          | データセット       | R2（RMSE）
------------------------------------------------------------------------------------------
PCE        | 補完       | SVM_Radial      | 500S+OH      | 0.757 （1.254）
PCE        | 未補完      | RandomForest    | 7変数+OH       | 0.735 （1.360）
Jsc        | 補完       | GPR_Linear      | 500S+OH      | 0.744 （2.398）
Jsc        | 未補完      | SVM_Radial      | 7変数+OH       | 0.771 （2.423）
Voc        | 補完       | RandomForest    | 7変数+FP       | 0.723 （0.081）
Voc        | 未補完      | SVM_Radial      | 7変数+FP       | 0.846 （0.060）
FF         | 補完       | kNN             | 500S+FP      | 0.587 （0.079）
FF         | 未補完      | kNN             | 500S+FP      | 0.539 （0.060）

[SUCCESS] 最適モデル表を作成しました:
/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/Corr_1000/Optimal_Model_Selection_Table_20260120.csv


In [3]:
import os
import pandas as pd
import numpy as np

# =============================================================================
# 設定：結果が格納されているベースパス
# =============================================================================
base_results_path = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/Corr_1000/results"

# 読み込むモデルディレクトリと対応名
model_configs = {
    "PLS":           "Corr_1000_PLS",
    "SVM_Linear":    "Corr_1000_SVM_Linear",
    "GPR_Linear":    "Corr_1000_GPR_Linear",
    "SVM_Radial":    "Corr_1000_SVM_Radial",
    "GPR_Radial":    "Corr_1000_GPR_Radial",
    "gcvEarth":      "Corr_1000_gcvEarth",
    "PPR":           "Corr_1000_PPR",
    "kNN":           "Corr_1000_kNN",
    "RandomForest":  "Corr_1000_RandomForest",
    "MLP":           "Corr_1000_MLP",
}

# -----------------------------------------------------------------------------
# 1. 全データの収集
# -----------------------------------------------------------------------------
all_results = []

for model_name, model_dir in model_configs.items():
    summary_path = os.path.join(base_results_path, model_dir, "all_summary_CV10.csv")
    
    if os.path.exists(summary_path):
        try:
            df = pd.read_csv(summary_path)
            for _, row in df.iterrows():
                # 補完有無の判定
                imp_type = "補完あり(m)" if row["File"].startswith("m_") else "未補完(n)"
                
                all_results.append({
                    "Target": row["Target"].strip(),
                    "Imp": imp_type,
                    "Model": model_name,
                    "Dataset": row["File"],
                    "R2": row.get("CV10_R2", 0),
                    "RMSE": row.get("CV10_RMSE", 999)
                })
        except Exception as e:
            print(f"Error loading {model_name}: {e}")

master_df = pd.DataFrame(all_results)

# -----------------------------------------------------------------------------
# 2. 条件ごとにソートして出力
# -----------------------------------------------------------------------------
print("\n" + "="*90)
print(f"{'Target':<10} | {'Type':<12} | {'Rank':<4} | {'Model':<15} | {'R2':<7} | {'RMSE':<8} | {'Dataset'}")
print("="*90)

targets = master_df["Target"].unique()
types = ["補完あり(m)", "未補完(n)"]

for tgt in targets:
    for t in types:
        # 該当条件の抽出
        sub = master_df[(master_df["Target"] == tgt) & (master_df["Imp"] == t)].copy()
        
        if sub.empty:
            continue
            
        # R2降順、RMSE昇順でソート
        sub = sub.sort_values(by=["R2", "RMSE"], ascending=[False, True])
        
        # コンソール出力
        for i, (idx, row) in enumerate(sub.iterrows(), 1):
            # 1位の行だけターゲット名を表示して視認性を高める
            tgt_disp = tgt if i == 1 else ""
            type_disp = t if i == 1 else ""
            
            print(f"{tgt_disp:<10} | {type_disp:<12} | {i:>2}: | {row['Model']:<15} | {row['R2']:>7.3f} | {row['RMSE']:>8.4f} | {row['Dataset']}")
        
        print("-" * 90)

print("\n[INFO] 各条件のトップモデルを確認してください。")


Target     | Type         | Rank | Model           | R2      | RMSE     | Dataset
Jsc        | 補完あり(m)      |  1: | GPR_Linear      |   0.744 |   2.3981 | m_all_OH_rebuilt.csv
           |              |  2: | SVM_Radial      |   0.734 |   2.4442 | m_all_OH_rebuilt.csv
           |              |  3: | SVM_Linear      |   0.732 |   2.4576 | m_all_OH_rebuilt.csv
           |              |  4: | SVM_Radial      |   0.724 |   2.5005 | m_all_FP_rebuilt.csv
           |              |  5: | PLS             |   0.712 |   2.5527 | m_all_OH_rebuilt.csv
           |              |  6: | RandomForest    |   0.710 |   2.5518 | m_base_FP_rebuilt.csv
           |              |  7: | RandomForest    |   0.709 |   2.5562 | m_all_FP_rebuilt.csv
           |              |  8: | SVM_Radial      |   0.702 |   2.6057 | m_base_OH_rebuilt.csv
           |              |  9: | MLP             |   0.701 |   2.5942 | m_all_FP_rebuilt.csv
           |              | 10: | kNN             |   0.694 |   2.624

In [7]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from zipfile import ZipFile

# =============================================================================
# 1. パスと設定
# =============================================================================
BASE_WORKING_DIR = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/Corr_1000"
IMPORTANCE_BASE_PATH = os.path.join(BASE_WORKING_DIR, "results")
RAW_DATA_PATH = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/"

OUTPUT_DIR_NAME = "Feature_Importance_Impact_Analysis_Results"
OUTPUT_DIR = os.path.join(BASE_WORKING_DIR, OUTPUT_DIR_NAME)
CSV_DIR = os.path.join(OUTPUT_DIR, "CSV_Summary")
for d in [os.path.join(OUTPUT_DIR, "PlanA_Directional"), 
          os.path.join(OUTPUT_DIR, "PlanB_SignLabeled"), CSV_DIR]:
    os.makedirs(d, exist_ok=True)

# 厳密なファイル名とモデルフォルダのキーワードを定義
# (ターゲット, 補完タイプ): (フォルダ検索語, 検索語2, 対応するRawデータ, 重要度CSV内のFile列の期待値)
best_model_configs = {
    ("PCEmax", "補完"):   ("svm", "radial", "m_all_OH_rebuilt.csv", "m_all_OH_rebuilt.csv"),
    ("PCEmax", "未補完"): ("random", "forest", "n_base_OH_rebuilt.csv", "n_base_OH_rebuilt.csv"),
    ("Jsc", "補完"):   ("gpr", "linear", "m_all_OH_rebuilt.csv", "m_all_OH_rebuilt.csv"),
    ("Jsc", "未補完"): ("svm", "radial", "n_base_OH_rebuilt.csv", "n_base_OH_rebuilt.csv"),
    ("Voc", "補完"):   ("random", "forest", "m_base_FP_rebuilt.csv", "m_base_FP_rebuilt.csv"),
    ("Voc", "未補完"): ("svm", "radial", "n_base_FP_rebuilt.csv", "n_base_FP_rebuilt.csv"),
    ("FF", "補完"):    ("knn", "", "m_all_FP_rebuilt.csv", "m_all_FP_rebuilt.csv"),
    ("FF", "未補完"):  ("knn", "", "n_all_FP_rebuilt.csv", "n_all_FP_rebuilt.csv")
}

# =============================================================================
# 2. データの収集と厳密なフィルタリング
# =============================================================================
all_data_list = []
if os.path.exists(IMPORTANCE_BASE_PATH):
    for d in os.listdir(IMPORTANCE_BASE_PATH):
        if not d.startswith("Corr_1000_"): continue
        dir_path = os.path.join(IMPORTANCE_BASE_PATH, d)
        
        # フォルダ内の重要度CSVをスキャン
        csv_files = [f for f in os.listdir(dir_path) if f.endswith(".csv") and "importance" in f.lower()]
        for f in csv_files:
            tmp_df = pd.read_csv(os.path.join(dir_path, f))
            tmp_df['Folder_Name'] = d.lower()
            # File列に基づいて、その行が補完か未補完かを厳密に判定
            tmp_df['Imputation_Type'] = tmp_df['File'].apply(lambda x: '補完' if str(x).startswith('m_') else '未補完')
            all_data_list.append(tmp_df)

master_df = pd.concat(all_data_list, ignore_index=True)

# =============================================================================
# 3. 解析・CSV・可視化
# =============================================================================
def get_rank_color(cmap_name, rank, total=15):
    return plt.get_cmap(cmap_name)(0.85 - (rank / total) * 0.55)

def get_impact_info(df_raw, target, feature):
    if df_raw is None or feature not in df_raw.columns: return 0, "Neutral"
    valid = df_raw[[feature, target]].dropna()
    if len(valid) < 2: return 0, "Neutral"
    v = valid[target][valid[feature]==1].mean() - valid[target][valid[feature]==0].mean() if set(valid[feature].unique()).issubset({0,1}) else valid[feature].corr(valid[target])
    return v, ("Positive" if v > 0.0001 else "Negative" if v < -0.0001 else "Neutral")

raw_data_cache = {}
targets = ["PCEmax", "Jsc", "Voc", "FF"]

for target in targets:
    dfs = {}
    for itype in ['補完', '未補完']:
        conf = best_model_configs.get((target, itype))
        if not conf: continue
        k1, k2, r_file, f_col = conf # f_col: CSV内のFile列と一致すべき名前
        
        # フォルダ名とFile列の両方で厳密にフィルタリング
        sub = master_df[(master_df['Target'] == target) & 
                        (master_df['Folder_Name'].str.contains(k1.lower())) & 
                        (master_df['Folder_Name'].str.contains(k2.lower())) & 
                        (master_df['File'] == f_col)].copy()
        
        if sub.empty: continue

        # Rawデータの取得
        if (target, itype) not in raw_data_cache:
            fpath = os.path.join(RAW_DATA_PATH, r_file)
            if os.path.exists(fpath): raw_data_cache[(target, itype)] = pd.read_csv(fpath)
        
        raw_df = raw_data_cache.get((target, itype))
        sub['Impact_Value'] = sub['Feature'].apply(lambda x: get_impact_info(raw_df, target, x)[0])
        sub['Trend'] = sub['Feature'].apply(lambda x: get_impact_info(raw_df, target, x)[1])
        
        sub = sub.sort_values('Importance', ascending=False)
        sub['Rank'] = range(1, len(sub) + 1)
        sub['Norm_Imp'] = sub['Importance'] / sub['Importance'].max()
        sub['Directional_Imp'] = sub['Norm_Imp'] * sub['Trend'].map({'Positive':1, 'Negative':-1, 'Neutral':0})

        # CSV出力
        sub.to_csv(os.path.join(CSV_DIR, f"{target}_{itype}_All.csv"), index=False)
        sub.head(15).to_csv(os.path.join(CSV_DIR, f"{target}_{itype}_Top15.csv"), index=False)
        dfs[itype] = sub.head(15)

    # 描画 (重複排除)
    if not dfs: continue
    all_feats = []
    seen = set()
    for f in list(dfs.get('補完', pd.DataFrame())['Feature']) + list(dfs.get('未補完', pd.DataFrame())['Feature']):
        if f not in seen:
            all_feats.append(f)
            seen.add(f)
    display_features = all_feats[::-1]

    for plan in ['A', 'B']:
        fig, ax = plt.subplots(figsize=(11, len(display_features) * 0.4 + 1.2))
        height = 0.35
        for i, feat in enumerate(display_features):
            for itype, cmap, offset in [('補完', 'Reds', height/2), ('未補完', 'Blues', -height/2)]:
                s_df = dfs.get(itype, pd.DataFrame())
                if not s_df.empty and feat in s_df['Feature'].values:
                    d = s_df[s_df['Feature'] == feat].iloc[0]
                    if plan == 'A':
                        ax.barh(i + offset, d['Directional_Imp'], height=height, color=get_rank_color(cmap, d['Rank']-1))
                    else:
                        ax.barh(i + offset, d['Norm_Imp'], height=height, color=get_rank_color(cmap, d['Rank']-1))
                        sign = "+" if d['Trend'] == "Positive" else "-" if d['Trend'] == "Negative" else ""
                        ax.text(d['Norm_Imp'] + 0.01, i + offset, sign, va='center', fontsize=16, fontweight='bold')
        ax.set_yticks(np.arange(len(display_features)))
        ax.set_yticklabels(display_features)
        ax.set_title(f"Target: {target}", loc='left')
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, "PlanA_Directional" if plan=='A' else "PlanB_SignLabeled", f"{target}_{plan}.png"), dpi=300)
        plt.close()

# ZIP圧縮
zip_path = os.path.join(BASE_WORKING_DIR, f"{OUTPUT_DIR_NAME}.zip")
with ZipFile(zip_path, 'w') as zipf:
    for root, _, files in os.walk(OUTPUT_DIR):
        for f in files: zipf.write(os.path.join(root, f), arcname=os.path.join(os.path.relpath(root, OUTPUT_DIR), f))

print(f"完了: {zip_path}")

完了: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/Corr_1000/Feature_Importance_Impact_Analysis_Results.zip


In [ ]:
import os
import pandas as pd

# =============================================================================
# 1. パスと設定
# =============================================================================
BASE_WORKING_DIR = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/Corr_1000"
IMPORTANCE_BASE_PATH = os.path.join(BASE_WORKING_DIR, "results")

# ターゲット、補完タイプ、フォルダキーワード、そして「厳密に一致すべきFile列名」
best_model_configs = {
    ("PCEmax", "補完"):   ("svm", "radial", "m_all_OH_rebuilt.csv"),
    ("PCEmax", "未補完"): ("random", "forest", "n_base_OH_rebuilt.csv"),
    ("Jsc", "補完"):   ("gpr", "linear", "m_all_OH_rebuilt.csv"),
    ("Jsc", "未補完"): ("svm", "radial", "n_base_OH_rebuilt.csv"),
    ("Voc", "補完"):   ("random", "forest", "m_base_FP_rebuilt.csv"),
    ("Voc", "未補完"): ("svm", "radial", "n_base_FP_rebuilt.csv"),
    ("FF", "補完"):    ("knn", "", "m_all_FP_rebuilt.csv"),
    ("FF", "未補完"):  ("knn", "", "n_all_FP_rebuilt.csv")
}

# =============================================================================
# 2. データの収集
# =============================================================================
all_data_list = []
if os.path.exists(IMPORTANCE_BASE_PATH):
    for d in os.listdir(IMPORTANCE_BASE_PATH):
        if not d.startswith("Corr_1000_"): continue
        dir_path = os.path.join(IMPORTANCE_BASE_PATH, d)
        csv_files = [f for f in os.listdir(dir_path) if f.endswith(".csv") and "importance" in f.lower()]
        for f in csv_files:
            tmp_df = pd.read_csv(os.path.join(dir_path, f))
            tmp_df['Folder_Name'] = d.lower()
            all_data_list.append(tmp_df)

if not all_data_list:
    print("Error: No importance data found.")
    exit()

master_df = pd.concat(all_data_list, ignore_index=True)

# =============================================================================
# 3. 各モデルのTop 20を表示
# =============================================================================
print("="*90)
print(f"{'Target':<8} | {'Type':<4} | {'Model Folder':<25} | {'Source File (Filter)':<25}")
print("-"*90)

for (target, itype), (k1, k2, f_col) in best_model_configs.items():
    # フォルダ名キーワードと、File列の完全一致でフィルタリング
    sub = master_df[(master_df['Target'] == target) & 
                    (master_df['Folder_Name'].str.contains(k1.lower())) & 
                    (master_df['Folder_Name'].str.contains(k2.lower())) & 
                    (master_df['File'] == f_col)].copy()
    
    print(f"{target:<8} | {itype:<4} | キーワード: {k1}+{k2:<12} | Filter: {f_col}")
    
    if sub.empty:
        print("  [!] 条件に一致するデータが見つかりませんでした。")
    else:
        # 重要度でソートしてTop 20を抽出
        top20 = sub.sort_values('Importance', ascending=False).head(20)
        
        # コンソール表示用の整形
        lines = []
        for i, (idx, row) in enumerate(top20.iterrows(), 1):
            lines.append(f"{i:2}. {row['Feature']:<15} ({row['Importance']:7.4f})")
        
        # 5個ずつ横に並べて表示（見やすさのため）
        for j in range(0, len(lines), 4):
            print("    " + "  ".join(lines[j:j+4]))
            
    print("-"*90)

print("="*90)